### Extraction of Employment & Wage Data

The goal of this notebook is to extract fiscal spending data from the Monthly Treasury Statement (MTS) at <a href="https://fiscaldata.treasury.gov/datasets/monthly-treasury-statement/summary-of-receipts-outlays-and-the-deficit-surplus-of-the-u-s-government">FiscalData.Treasury.Gov</a>  with BeautifulSoup, saving it to a dataframe and then a CSV file.

The direction of this exercise follows Chapter 5 of "The Data Wrangling Workship" second edition on the "Introduction to Beautiful Soup 4 and Web Page Parsing". 

Due to the nature of data tables in the wild, there are several classes that separate us from the data, unlike book's example. Therefore slight changes such as using the `.get_text()` method have been added for efficiency.

### Extracting data from HTML file

In [1]:
from bs4 import BeautifulSoup


In [2]:
# Open the file
with open("/home/jeremy/Documents/GitHub/DSC540/TermProject/data/Monthly Treasury Statement.html") as mts:
    soup = BeautifulSoup(mts)
    print(type(soup))

<class 'bs4.BeautifulSoup'>


You can pass a file handler directly to the constructor of the BeautifulSoup
object and it will read the contents from the file that the handler is attached to.

Ensure consistent indentation with .prettyify from BeautifulSoup.

In [3]:
# print(soup.prettify)

Dereference the `<p>` tag as a mamaber variable of the BeautifuilSoup.

In [4]:
print(soup.p)

<p>The Monthly Treasury Statement (MTS) dataset provides information on the flow of money into and out of the U.S. Department of the Treasury. It includes how deficits are funded, such as borrowing from the public or reducing operating cash, and how surpluses are distributed. Further tables categorize spending (outlays) by department and agency, revenue (receipts) by specific taxes or other government sources of income, and transactions with trust funds such as Social Security or Medicare. All values are reported in millions of U.S. dollars.</p>


This finds just the first `<p>` tag, but to find them all, `findall` is used.

In [5]:
all_ps = soup.find_all('p')
print(all_ps)

[<p>The Monthly Treasury Statement (MTS) dataset provides information on the flow of money into and out of the U.S. Department of the Treasury. It includes how deficits are funded, such as borrowing from the public or reducing operating cash, and how surpluses are distributed. Further tables categorize spending (outlays) by department and agency, revenue (receipts) by specific taxes or other government sources of income, and transactions with trust funds such as Social Security or Medicare. All values are reported in millions of U.S. dollars.</p>, <p>The Monthly Treasury Statement (MTS) dataset provides information on the flow of money into and out of the U.S. Department of the Treasury. It includes how deficits are funded, such as borrowing from the public or reducing operating cash, and how surpluses are distributed. Further tables categorize spending (outlays) by department and agency, revenue (receipts) by specific taxes or other government sources of income, and transactions with 

Up until now only specific tags have been used to extrct data from webpages. BeautifulSoup also allows us to extract tags of the same type. Below is an example of using the `table` method to pull tags related to tables on the web. `<tr>`, `<td>`, `<th>`...

In [6]:
table = soup.table
print(table.get_text(separator="\n", strip=True))

# .get_text() extracts all non-tag text from the HTML element
    #   It allows us to add separators that will print
    #       the text in a far more readable format.

Name
Date
Size
Monthly Treasury State
ment.pdf
December 2025
3 MB
Download
Monthly Receipts, Outlays, and Deficit or Sur
plus.pdf
December 2025
104 KB
Download
Monthly Receipts, Outlays, and Deficit or Sur
plus.xls
December 2025
92 KB
Download
Distributed Offsetting Receipts by Department Re
port.pdf
December 2025
164 KB
Download
Distributed Offsetting Receipts by Department Report
(.xlsx)
December 2025
282 KB
Download
Budget Summary with Adjusted Differences Re
port.pdf
December 2025
701 KB
Download


The above data is only the immediate table found in the HTML. It is the table that contains the downloads for PDF files of the data.

Browsing all the possible children of a tag, and not only the immediate one. This is done using the `decendants` generator form bs4

In [7]:
children = table.children
des = table.descendants
print(len(list(children)), len(list(des)))

2 130


The above is the difference between children and descendents.

In the first attempt to extract data from the data table, only the first table was returned. This is class being accessed contains multiple tables. Inspecting the page can help identify the right parts of a webpage to pull the correct information. This is a more direct approach, as there are significantly more descendants the Treasury Data than the book's example.

Tables can be held in `classes` which can be accessed to find data tables.

Now we will use the .select() method from `bs4` to search inside a DIV for tables.

In [8]:
# Inside the .select() is where the div we want to search
#   for is placed.
tables = soup.select(
    "div.data-table-module--overlayContainerNoFooter--a46b8 table"
)

table = []
# Save output for use in next step
# Creating a loop to pick up all tables in the DIV
for table in tables:
    
     table.append(table.prettify())

# Making "table" equal to tables[1] will extract the
#   uneeded download file table.
table = tables[0]
# print(table)


In [9]:
# Check the new table data
# print(table.get_text(separator="\n", strip=True))

There are two tables in the `<div>` we're looking at. The trasury data is in the first of the two, so output should = `output[0]`, since `table[0]` is the true data we're searching for. The above was an operation to check where the true data is. The other table are just file download links.

### Saving to a Dataframe

Install pandas

In [10]:
import pandas as pd

Separate headers from the data.

In [31]:
rows = table.find_all('tr')[1:]
# Correct: headers is a <tr> tag
headers = table.find('tr')  # do NOT call getText() here
col_headers = [th.getText() for th in headers.findAll('th')]

col_headers

/tmp/ipykernel_44697/1751528950.py:4: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  col_headers = [th.getText() for th in headers.findAll('th')]


['Record Datemm/dd/yyyy - mm/dd/yyyy',
 'Parent ID\u200b',
 'Classification ID\u200b',
 'Classification Description\u200b',
 'Current Month Gross Receipts Amount\u200b',
 'Current Month Gross Outlay Amount\u200b',
 'Current Month Deficit Surplus Amount\u200b',
 'Table Number\u200b',
 'Source Line Number\u200b',
 'Print Order Sequence Number\u200b',
 'Line Code Number\u200b',
 'Data Type Code\u200b',
 'Record Type Code\u200b',
 'Sequence Level Number\u200b',
 'Sequence Number Code\u200b',
 'Fiscal Year\u200b',
 'Fiscal Quarter Number\u200b',
 'Calendar Year\u200b',
 'Calendar Quarter Number\u200b',
 'Calendar Month Number\u200b',
 'Calendar Day Number\u200b']

In [32]:
data_rows = []

# Clean row data to only include text inside 'td' or 'th' tags.
for row in rows:
    
    cells = [cell.get_text(strip=True) for cell in row.find_all(['td', 'th'])]
    data_rows.append(cells)

data_rows

[['12/31/2025',
  '',
  '57633926',
  'FY 2025',
  '',
  '',
  '',
  '1',
  '1',
  '1',
  '10',
  'S',
  'SL',
  '1',
  '1',
  '2026',
  '1',
  '2025',
  '4',
  '12',
  '31'],
 ['12/31/2025',
  '57633926',
  '57633931',
  'October',
  '$326,770,236,058.10',
  '$584,220,273,025.31',
  '$257,450,036,967.21',
  '1',
  '2',
  '2',
  '20',
  'D',
  'MTH',
  '2',
  '1.1',
  '2026',
  '1',
  '2025',
  '4',
  '12',
  '31'],
 ['12/31/2025',
  '57633926',
  '57633935',
  'November',
  '$301,754,299,812.79',
  '$668,517,135,060.47',
  '$366,762,835,247.68',
  '1',
  '3',
  '3',
  '30',
  'D',
  'MTH',
  '2',
  '1.2',
  '2026',
  '1',
  '2025',
  '4',
  '12',
  '31'],
 ['12/31/2025',
  '57633926',
  '57633940',
  'December',
  '$454,414,708,491.47',
  '$541,146,327,504.76',
  '$86,731,619,013.29',
  '1',
  '4',
  '4',
  '40',
  'D',
  'MTH',
  '2',
  '1.3',
  '2026',
  '1',
  '2025',
  '4',
  '12',
  '31'],
 ['12/31/2025',
  '57633926',
  '57633942',
  'January',
  '$513,294,342,651.93',
  '$641,9

Carefully place the row data into a pandas dataframe.

In [34]:
df_data = [[td.getText() for td in tr.findAll('td')] \
            for tr in rows]
df_data

/tmp/ipykernel_44697/3341554793.py:1: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  df_data = [[td.getText() for td in tr.findAll('td')] \


[['12/31/2025',
  '',
  '57633926',
  'FY 2025',
  '',
  '',
  '',
  '1',
  '1',
  '1',
  '10',
  'S',
  'SL',
  '1',
  '1',
  '2026',
  '1',
  '2025',
  '4',
  '12',
  '31'],
 ['12/31/2025',
  '57633926',
  '57633931',
  'October',
  '$326,770,236,058.10',
  '$584,220,273,025.31',
  '$257,450,036,967.21',
  '1',
  '2',
  '2',
  '20',
  'D',
  'MTH',
  '2',
  '1.1',
  '2026',
  '1',
  '2025',
  '4',
  '12',
  '31'],
 ['12/31/2025',
  '57633926',
  '57633935',
  'November',
  '$301,754,299,812.79',
  '$668,517,135,060.47',
  '$366,762,835,247.68',
  '1',
  '3',
  '3',
  '30',
  'D',
  'MTH',
  '2',
  '1.2',
  '2026',
  '1',
  '2025',
  '4',
  '12',
  '31'],
 ['12/31/2025',
  '57633926',
  '57633940',
  'December',
  '$454,414,708,491.47',
  '$541,146,327,504.76',
  '$86,731,619,013.29',
  '1',
  '4',
  '4',
  '40',
  'D',
  'MTH',
  '2',
  '1.3',
  '2026',
  '1',
  '2025',
  '4',
  '12',
  '31'],
 ['12/31/2025',
  '57633926',
  '57633942',
  'January',
  '$513,294,342,651.93',
  '$641,9

In [36]:
df = pd.DataFrame(df_data, columns=col_headers)
df.head()

,Record Datemm/dd/yyyy - mm/dd/yyyy,Parent ID​,Classification ID​,Classification Description​,Current Month Gross Receipts Amount​,Current Month Gross Outlay Amount​,Current Month Deficit Surplus Amount​,Table Number​,Source Line Number​,Print Order Sequence Number​,...,Data Type Code​,Record Type Code​,Sequence Level Number​,Sequence Number Code​,Fiscal Year​,Fiscal Quarter Number​,Calendar Year​,Calendar Quarter Number​,Calendar Month Number​,Calendar Day Number​
0,12/31/2025,,57633926,FY 2025,,,,1,1,1,...,S,SL,1,1,2026,1,2025,4,12,31
1,12/31/2025,57633926,57633931,October,"$326,770,236,058.10","$584,220,273,025.31","$257,450,036,967.21",1,2,2,...,D,MTH,2,1.1,2026,1,2025,4,12,31
2,12/31/2025,57633926,57633935,November,"$301,754,299,812.79","$668,517,135,060.47","$366,762,835,247.68",1,3,3,...,D,MTH,2,1.2,2026,1,2025,4,12,31
3,12/31/2025,57633926,57633940,December,"$454,414,708,491.47","$541,146,327,504.76","$86,731,619,013.29",1,4,4,...,D,MTH,2,1.3,2026,1,2025,4,12,31
4,12/31/2025,57633926,57633942,January,"$513,294,342,651.93","$641,934,549,325.42","$128,640,206,673.49",1,5,5,...,D,MTH,2,1.4,2026,1,2025,4,12,31


### Exporting as an Excel File

From placing it in a dataframe, the data will be exported to a excel file through the use of `openpyxl`, which is a Python Library for reading/writing Excel 2010 xlsx/xslm/xltx/xltm files.

In [38]:
# !pip install openpyxl

In [46]:
# Define output path
output_path = '/home/jeremy/Documents/GitHub/DSC540/TermProject/data/MTS_Spending_Data.xlsx'

In [53]:
writer = pd.ExcelWriter(output_path)
df.to_excel(writer, sheet_name='Sheet1')
writer.close()  # explicitly save
